# Bitcoin Market Sentiment & Hyperliquid Trader Performance Analysis
This notebook explores the relationship between trader performance (transaction logs from Hyperliquid) and Bitcoin market sentiment (Fear & Greed Index). 

Our objectives are:
1. **Load and Clean** the datasets to establish a 2-year overlapping window.
2. **Aggregate and Profile** the 32 unique traders into performance, size, and activity cohorts.
3. **Merge and Map** transactions across daily market conditions (Extreme Fear, Fear, Neutral, Greed, Extreme Greed).
4. **Analyze Behavior** (such as Buying vs. Selling Balance) to identify trading behavior patterns.
5. **Synthesize Insights** to drive smarter execution strategies.

## 1. Environment & Ingestion Setup
First, we append the source directory to our system path, load our standard configurations, and load the raw datasets.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root to sys.path
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_fear_greed_data, load_trader_data
from src.metrics import calculate_trader_metrics, calculate_coin_metrics
from src.sentiment import merge_trader_and_sentiment_data, analyze_by_sentiment_regime
from src.visualizer import generate_and_save_plots
from src import config

# Load clean datasets
fg_df = load_fear_greed_data()
trader_df = load_trader_data()

print(f"Fear & Greed Data: {fg_df.shape[0]} daily records.")
print(f"Hyperliquid Trader Logs: {trader_df.shape[0]} transactions.")

## 2. Trader Performance & Profiling
We aggregate individual transactions to build trader profiles. We compute Profit (PnL), volume, fees, win rates, and profit factors per account, and bin them into volume cohorts (`Whale`, `Medium`, `Retail`) and activity levels.

In [ ]:
trader_metrics = calculate_trader_metrics(trader_df)

print("--- Cohort Distributions ---")
print("Volume Cohorts:\n", trader_metrics['volume_cohort'].value_counts())
print("\nProfitability Cohorts:\n", trader_metrics['profit_cohort'].value_counts())

print("\n--- Top 5 Profitable Traders (by Net Profit) ---")
display(trader_metrics.sort_values(by='net_pnl', ascending=False).head(5))

## 3. Market Sentiment & Market Conditions Mapping
We merge daily Fear & Greed classifications with transaction dates. We then analyze overall trading activity and cohort metrics across different market conditions.

In [ ]:
merged_df = merge_trader_and_sentiment_data(trader_df, fg_df)
regime_results = analyze_by_sentiment_regime(merged_df, trader_metrics)

print("--- Overall Metrics by Sentiment and Market Conditions ---")
display(regime_results['overall'][['classification', 'trade_count', 'total_volume', 'net_pnl', 'win_rate']])

## 4. Visualizations
We generate and inspect the four standard visual figures to trace patterns.

In [ ]:
# Generate the plots
generate_and_save_plots(merged_df, trader_metrics, regime_results)

# Display plots inline using IPython display
from IPython.display import Image, display

print("\n--- 1. Sentiment and Market Conditions Distribution ---")
display(Image(filename=os.path.join(config.PLOTS_DIR, "regime_distribution.png")))

print("\n--- 2. Cohort Performance by Market Conditions ---")
display(Image(filename=os.path.join(config.PLOTS_DIR, "cohort_performance_regime.png")))

print("\n--- 3. Cumulative Net Profit Over Time ---")
display(Image(filename=os.path.join(config.PLOTS_DIR, "cumulative_pnl_over_time.png")))

print("\n--- 4. Net Buying Activity (Buying vs. Selling Balance) ---")
display(Image(filename=os.path.join(config.PLOTS_DIR, "net_buy_ratio_regime.png")))

## 5. Summary of Key Findings

### 1. Difference Between Large and Small Traders
*   **How Small Traders Behave in Different Market Conditions**: Small traders trade much less when the market is panicking (only 871 trades during `Extreme Fear`) compared to when things are going up (`Greed`: 1,975 trades). When they do try to trade during extreme panic, their win rate drops to a low **42.9%**, showing they struggle to time the market bottoms.
*   **How Large Traders Behave in Different Market Conditions**: Large traders keep execution active no matter what. They made over 13,000 trades during Extreme Fear and kept a high win rate of **79.8%**. They also made their biggest absolute profits (Net Profit) during `Extreme Greed` (+$1.67M), taking advantage of high market excitement.

### 2. Trading Behavior Patterns
*   **Following Market Trends & Poor Performance During Market Fear**: During Extreme Fear, retail traders are mostly buying (a positive Buying vs. Selling Balance of +0.36), but since their win rate is so low, they are likely buying assets that are still falling. During greedy markets, they net sell (-0.63), showing they take profit too early and miss out on bigger trends.
*   **Consistent Market-Making Strategy & Balanced Trading Strategy**: The most interesting finding is that the large traders have a Buying vs. Selling Balance (Net Buy Ratio) very close to zero (-0.03 to +0.06) across all market conditions. This means their buys and sells are almost perfectly equal. They are likely running automated market-making algorithms that don't try to predict where the price is going, but instead provide liquidity and collect trading fees.